In [89]:
import sys
sys.path.append("/home/landelle/pytorch-cifar/models")
import util
import resnet
import numpy as np
import torch
import torchvision


In [90]:

transform = torchvision.transforms.transforms.Compose([
    torchvision.transforms.transforms.ToTensor()
])

trainset = torchvision.datasets.CIFAR10(root='datasets', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='datasets', train=False, download=True, transform=transform)

trainset, testset

Files already downloaded and verified
Files already downloaded and verified


(Dataset CIFAR10
     Number of datapoints: 50000
     Root location: datasets
     Split: Train
     StandardTransform
 Transform: Compose(
                ToTensor()
            ), Dataset CIFAR10
     Number of datapoints: 10000
     Root location: datasets
     Split: Test
     StandardTransform
 Transform: Compose(
                ToTensor()
            ))

In [91]:

BATCH_SIZE = 50
N_BATCHES_IN_TRAIN_SET = len(trainset) // BATCH_SIZE
N_BATCHES_IN_TEST_SET = len(testset) // BATCH_SIZE

NUM_WORKERS = 8

# Fixed learning rate
LR = 0.05 #0.005 LR=0.1 is used by kuangliu with his implementation of ResNet18 for epochs [0;150)

# CIFAR10 number of classes
NUM_CLASS = 10

# CIFAR10 image metadata
CHANNEL, IMAGE_SIZE, _ = trainset[0][0].shape
print("images are:", IMAGE_SIZE, CHANNEL)

TRAIN_EPOCHS = 5
print(N_BATCHES_IN_TRAIN_SET)

images are: 32 3
1000


In [92]:

trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
testloader = torch.utils.data.DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


In [93]:

def get_model():
    """ Get model is used to remake the model between lambda attempts """
    return resnet.ResNet18()

def mixup_data_nmix(n_ways, X, y, lam_):
    """
    Mixes data n ways with itself shuffled without overlaps, returns X_mixed and an array of every y shuffling
    Example:
        mixup_data(3, X, y, (.5, .5, 0)) will return X_mixed, y[permutation0], y[permutation1], y[permutation2]
        and X_mixed will veriy X_mixed = .5*X[permutation0] + .5*X[permutation1] + 0*X[permutation2]
    X: tensor of shape (batch_size, w, h, 3)
    y: tensor of shape (batch_size,)
    lam_: tensor of shape (n_ways, 1, 1, 1, 1)
    """
    assert lam_.shape[0] == n_ways
    assert X.shape[0] == y.shape[0]
    assert lam_.sum() == 1
    perms = util.no_overlap_perms_random(n_ways, X.shape[0])
    #X_permutations = np.array([X[perm].numpy() for perm in perms])
    X_permutations = torch.stack([X[perm] for perm in perms], 0)
    #reshape
    #lam_rs_tensor = torch.from_numpy(lam_rs).float()
    #X_mixed = (X_permutations * lam_rs_tensor).sum(axis=0)
    X_mixed = (X_permutations * lam_).sum(axis=0)
    ys = [y[perm] for perm in perms]
    #return torch.from_numpy(X_mixed).float(), ys, perms
    return X_mixed, ys, perms

def train_nmix(device, model, optimizer, criterion, lam_vec, trainloader, *, 
          n_epochs=None, n_batches=None,
          save_raw=False, save_state_dicts="", save_tensorboard="",
         LR=.01, TRAIN_EPOCHS=50, N_BATCHES_IN_TRAIN_SET=500):
    """
    save_raw: save raw results (also print to external file) 
    save_state_dicts: No|Override|Separate, saves state_dicts after an epoch
    save_tensorboard: saves tensorboard data
    """
    print_ = util.get_print(save_raw)
    NMIX = lam_vec.shape[0]
    print_("Performing n-mixup with N-mix:", NMIX)
    
    # switch to train mode
    model.train()
    n_epochs = n_epochs if n_epochs else TRAIN_EPOCHS
    for epoch in range(1, n_epochs + 1):
        print_("Epoch[{}/{}]".format(epoch, n_epochs))
        for batch_id, (images, labels) in enumerate(trainloader):
            if n_batches and batch_id > n_batches:
                break
            labels, images = labels.to(device), images.to(device)
            images, ys, perms = mixup_data_nmix(NMIX, images, labels, lam_vec)
            #outputs = model(images.float())
            outputs = model(images)
            loss = mixup_criterion_nmix(criterion, outputs, ys, lam_vec)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if batch_id % 50 == 0:
                print_('Loss :{:.4f} Epoch[{}/{}] Batch[{}/{}] batch_shape:{}'.format(
                    loss.item(), epoch, n_epochs, batch_id, N_BATCHES_IN_TRAIN_SET, images.shape))
        # Each epoch if enabled: save state dicts
        if save_state_dicts=="Separate":
            torch.save(model.state_dict(), STATE_DICTS_DIR + STATE_DICTS_PREFIX + "_epoch_" + str(epoch))
        elif save_state_dicts=="Override":
            torch.save(model.state_dict(), STATE_DICTS_DIR + STATE_DICTS_PREFIX + "_override")

def collect_results_nmix(trainloader, testloader, lambdas, *, model_getter=get_model, n_epochs=10, n_batches=None, half=False, no_mixup=False,
                    save_raw=False, save_state_dicts=False, save_tensorboard=False, 
         LR=.01, TRAIN_EPOCHS=50, N_BATCHES_IN_TRAIN_SET=500, use_cuda=True):
    """
    n_batches: None trains on whole dataset otherwise it trains on n_batches of BATCH_SIZE per epoch    
    
    """
    USE_CUDA = use_cuda
    print_ = util.get_print(save_raw)

    test_accs = {}
    for lambda_ in lambdas:
        if half and lambda_>.5:
            break
        if no_mixup and lambda_>0:
            break
        print_("Trying lambda=", lambda_)        

        device = torch.device('cuda' if USE_CUDA else 'cpu')
        model = model_getter().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        criterion = torch.nn.CrossEntropyLoss()
        
        lam_rs = lambda_.reshape((-1, 1, 1, 1, 1))
        lam_rs_tensor = torch.from_numpy(lam_rs).float().to(device)
        
        train_nmix(device, model, optimizer, criterion, lam_rs_tensor, trainloader,
              n_epochs=n_epochs, n_batches=n_batches,
              save_raw=save_raw, save_state_dicts=save_state_dicts, save_tensorboard=save_tensorboard,
             LR=LR, TRAIN_EPOCHS=TRAIN_EPOCHS, N_BATCHES_IN_TRAIN_SET=N_BATCHES_IN_TRAIN_SET)
        test_accs[str(lambda_)] = util.test(device, model, testloader, save_raw=save_raw)

        del device, model, optimizer, criterion
    
    return test_accs

def mixup_criterion_nmix(criterion, preds, ys, lambda_v):
    return sum(lambda_v[i] * criterion(preds, ys[i]) for i in range(lambda_v.shape[0]))


In [94]:
lam_try = np.array([1., .5, .25])
lam_try_normalized = lam_try / lam_try.sum()


In [96]:

LAMBDAS = np.array([[1., 0., 0.], lam_try_normalized])
print(LAMBDAS)

SAVE_RAW = True
if SAVE_RAW:
    with open("results", "w") as f: pass
test_accs2 = collect_results_nmix(trainloader, testloader, LAMBDAS, model_getter=get_model,
                                  n_epochs=5, n_batches=500,
                                  save_raw=SAVE_RAW, use_cuda=True)

[[1.         0.         0.        ]
 [0.57142857 0.28571429 0.14285714]]
Trying lambda= [1. 0. 0.]
Performing n-mixup with N-mix: 3
Epoch[1/5]
Loss :2.3261 Epoch[1/5] Batch[0/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :2.1555 Epoch[1/5] Batch[50/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :1.9107 Epoch[1/5] Batch[100/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :2.0153 Epoch[1/5] Batch[150/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :1.9635 Epoch[1/5] Batch[200/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :1.9707 Epoch[1/5] Batch[250/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :1.6143 Epoch[1/5] Batch[300/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :1.6469 Epoch[1/5] Batch[350/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :1.4375 Epoch[1/5] Batch[400/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :1.8098 Epoch[1/5] Batch[450/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :1.4572 Epoch[1/5] Batch[500/500] batch_shape:torch.Size([50, 3, 

Loss :2.0239 Epoch[5/5] Batch[50/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :1.8728 Epoch[5/5] Batch[100/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :1.8944 Epoch[5/5] Batch[150/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :1.8748 Epoch[5/5] Batch[200/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :1.9226 Epoch[5/5] Batch[250/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :1.9299 Epoch[5/5] Batch[300/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :1.9905 Epoch[5/5] Batch[350/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :1.9711 Epoch[5/5] Batch[400/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :1.8968 Epoch[5/5] Batch[450/500] batch_shape:torch.Size([50, 3, 32, 32])
Loss :1.9249 Epoch[5/5] Batch[500/500] batch_shape:torch.Size([50, 3, 32, 32])
Test Accuracy of the model on the test images: 45.07 %
